# Classify gender

## Qué hace

Cuenta cuántos usuarios parecen hombres, mujeres o de género no definido, mirando solo su nombre.

## Qué pasa en la base de datos

Solo lee la tabla `users`. No cambia nada.

## Qué comando usa

`clients/terminal/classifyGender.txt`. El bloque de abajo llama a la misma función, `classifyGender()`.

## Anotaciones

- ⚠️ Necesita Ollama en marcha. Si está apagado, se ve `Error: Users could not be classified: <motivo>`.
- Cada nombre se lo pregunta a un modelo de IA local, así que puede tardar unos segundos.
- Pulsa ejecutar en el bloque de abajo. No hace falta escribir código.

In [1]:
import os
import subprocess
from pathlib import Path

# Sube de carpeta hasta encontrar la app (la carpeta que contiene logic/).
app = Path.cwd()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent
if not (app / "logic").is_dir():
    raise SystemExit("App not found.")

code = r"""
import Table from "cli-table3";
import { classifyGender } from "./logic/classifyGender.js";

try {
  const counts = await classifyGender();

  const table = new Table({
    head: ["Men", "Women", "Undefined"],
    style: { head: ["cyan", "bold"] },
  });
  table.push([counts.men, counts.women, counts.undefinedGender]);
  console.log(table.toString());
} catch (error) {
  console.error(`Error: ${error.message}`);
  process.exit(1);
}
"""

# Fuerza los colores del encabezado aunque la salida no sea una terminal.
env = os.environ.copy()
env["FORCE_COLOR"] = "1"
env.pop("NO_COLOR", None)

# La salida se lee en UTF-8 para que los bordes de la tabla no fallen en Windows.
completed = subprocess.run(
    ["node", "--no-warnings", "--input-type=module", "-e", code],
    cwd=app,
    capture_output=True,
    text=True,
    encoding="utf-8",
    env=env,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    print(completed.stderr, end="")

┌─────┬───────┬───────────┐
│ Men │ Women │ Undefined │
├─────┼───────┼───────────┤
│ 1   │ 1     │ 0         │
└─────┴───────┴───────────┘
